*Notebook trước: [5. Data Visualization](./5.%20Data%20Visualization.ipynb)*

# Bước 6: Model Building & Evaluation — Tối Giản 2 Bảng Chuẩn
### Đề án: Phân Tích Hành Vi Mua Hàng Trong Thương Mại Điện Tử (ADY201m)
---
### Thiết Lập Bài Toán & Lựa Chọn Đặc Trưng
* **Biến mục tiêu:** PurchaseStatus (Khách hàng có chốt đơn mua hàng hay không: 1 = Mua, 0 = Không mua).
* **Biến đầu vào:** LastPurchaseDaysAgo, LoyaltyProgram, AnnualIncome, DiscountsAvailed, Age.
* **Lý do chọn:**
  - **Thời gian từ lần mua cuối (LastPurchaseDaysAgo)** phản ánh độ gắn kết và mức độ quan tâm gần đây của khách — khách mới mua gần đây có xu hướng mua tiếp rất cao, còn khách lâu ngày không quay lại thường đã rời bỏ sàn.
  - **Thẻ hội viên (LoyaltyProgram)** và **Thu nhập (AnnualIncome)** đại diện cho độ trung thành và khả năng tài chính chi phối trực tiếp đến quyết định chi tiền.
  - **Số mã giảm giá (DiscountsAvailed)** và **Độ tuổi (Age)** được thêm vào để kiểm tra xem thói quen săn khuyến mãi và nhóm tuổi có đóng góp thêm thông tin vào quyết định chốt đơn hay không.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Nạp dữ liệu sạch & chọn các biến cốt lõi từ Tệp 3-4-5
df = pd.read_csv('../Data/customerData_clean.csv' if pd.io.common.file_exists('../Data/customerData_clean.csv') else 'Data/customerData_clean.csv')
features = ['Age', 'AnnualIncome', 'LastPurchaseDaysAgo', 'DiscountsAvailed', 'LoyaltyProgram']
X_train, X_test, y_train, y_test = train_test_split(df[features], df['PurchaseStatus'], test_size=0.2, random_state=42)

In [2]:
# 2. Huấn luyện 2 mô hình & xuất Bảng So Sánh Hiệu Năng
dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42).fit(X_train, y_train)

models_df = pd.DataFrame({
    'Mô hình': ['Decision Tree', 'Random Forest'],
    'Accuracy': [round(accuracy_score(y_test, dt.predict(X_test)), 3), round(accuracy_score(y_test, rf.predict(X_test)), 3)],
    'F1-Score': [round(f1_score(y_test, dt.predict(X_test)), 3), round(f1_score(y_test, rf.predict(X_test)), 3)]
})
display(models_df)

,Mô hình,Accuracy,F1-Score
0,Decision Tree,0.874,0.855
1,Random Forest,0.881,0.874


In [3]:
# 3. Xuất Bảng Độ Quan Trọng Của Biến (Feature Importance)
fi_df = pd.DataFrame({
    'Biến': features,
    'Độ quan trọng': [round(x, 3) for x in rf.feature_importances_]
}).sort_values('Độ quan trọng', ascending=False)
display(fi_df)

,Biến,Độ quan trọng
2,LastPurchaseDaysAgo,0.946
4,LoyaltyProgram,0.026
1,AnnualIncome,0.020
0,Age,0.006
3,DiscountsAvailed,0.003


In [ ]:
# 4. Phân bổ 3 Kịch bản can thiệp kinh doanh dựa trên xác suất P
prob = rf.predict_proba(X_test)[:, 1]
n = len(prob)
c1, c2, c3 = (prob >= 0.70).sum(), ((prob >= 0.30) & (prob < 0.70)).sum(), (prob < 0.30).sum()

print(f"Kịch bản 1 (P >= 70%): {c1:,} khách ({c1/n*100:.1f}%) -> Giữ nguyên giá niêm yết (Không cấp voucher)")
print(f"Kịch bản 2 (30% <= P < 70%): {c2:,} khách ({c2/n*100:.1f}%) -> Tặng voucher kích cầu kịp thời (Tối đa 5 mã)")
print(f"Kịch bản 3 (P < 30%): {c3:,} khách ({c3/n*100:.1f}%) -> Gửi Email hoặc trên các kênh tiếp thị chăm sóc tặng Hội viên 1 tháng nếu quay lại (Tối đa 2 lần/năm)")

Kịch bản 1 (P >= 70%): 32,268 khách (37.3%) -> Giữ nguyên giá niêm yết (Không cấp voucher)
Kịch bản 2 (30% <= P < 70%): 14,504 khách (16.8%) -> Tặng voucher kích cầu kịp thời (Tối đa 5 mã)
Kịch bản 3 (P < 30%): 39,680 khách (45.9%) -> Gửi Email chăm sóc tặng Hội viên 1 tháng nếu quay lại (Tối đa 2 lần/năm)


---
## Đúc Kết Nghiệp Vụ & Kế Hoạch Can Thiệp 3 Kịch Bản
1. **Mô hình tối ưu:** Random Forest cho hiệu năng nhỉnh hơn nhẹ so với Decision Tree (`Accuracy = 0.881`, `F1-Score = 0.874`).
2. **Đặc trưng chi phối:** Biến `LastPurchaseDaysAgo` chiếm hơn **94.6%** độ quan trọng, kết hợp cùng `LoyaltyProgram` (2.6%) và `AnnualIncome` (2.0%) làm đòn bẩy kích cầu.
3. **Thực thi 3 Kịch bản kinh doanh (Lấy khách hàng làm trọng tâm):
   * **Kịch bản 1 ($P \ge 70\%$ — Khách tự mua):** Giữ nguyên giá niêm yết, không cấp voucher để tối đa hóa biên lợi nhuận.
   * **Kịch bản 2 ($30\% \le P < 70\%$ — Khách lưỡng lự):** Tặng voucher kích cầu kịp thời, chặn cứng tối đa 5 mã/khách theo điểm bão hòa SQL 6.
   * **Kịch bản 3 ($P < 30\%$ — Khách > 60 ngày không mua / Nguy cơ bỏ sàn):
     * *Chiến lược chăm sóc tái tương tác (Win-back):* Không bỏ mặc người dùng mà chủ động tiếp cận qua kênh chăm sóc thân thiện (gửi Email cá nhân hóa hoặc thông báo ứng dụng).
     * *Chính sách ưu đãi quay lại:* Tặng ngay quyền lợi **Hội viên (`LoyaltyProgram`) miễn phí trong 1 tháng** nếu khách quay trở lại mua sắm.
     * *Kiểm soát rủi ro:* Giới hạn tối đa **1 năm chỉ áp dụng 2 lần/khách** để bảo đảm tính bền vững của chính sách.
